# Looking at NDV1 with pandas

NDV1 is the original nested JSON document. pandas helps us make simple tables from parts of NDV1 so we can inspect and filter the document more easily.

In [ ]:
import json
from pathlib import Path

import pandas as pd

In [ ]:
# Find the repository root whether the notebook runs from the repo root
# or from the dataset folder.
repository_root = Path.cwd()
if not (repository_root / "project's goal").exists():
    repository_root = repository_root.parent

path = (
    repository_root
    / "project's goal"
    / "example"
    / "ndv1-safara-professional-example.json"
)

data = json.loads(path.read_text(encoding="utf-8"))
ndv1 = data["normalizedDocument"]

print(f"Loaded: {path.resolve()}")
print(f"NDV1 version: {ndv1['version']}")
print(f"Number of pages: {len(ndv1['pages'])}")

## First, look at the original NDV1 structure

The following cell shows the first page and its first text blocks. This is the original document data before pandas reshapes anything.

In [ ]:
first_page = ndv1["pages"][0]

print("Page keys:", list(first_page.keys()))
print("First three text blocks:")
first_page["textBlocks"][:3]

## Make a text-block dataset

This creates one row for every `textBlock` in NDV1. The IDs and page numbers are preserved so each row can be traced back to the original document.

In [ ]:
text_rows = []

for page in ndv1["pages"]:
    for block in page.get("textBlocks", []):
        box = block.get("boundingBox", {})

        text_rows.append({
            "page": page["number"],
            "block_id": block["id"],
            "kind": block.get("kind"),
            "text": block["text"],
            "x": box.get("x"),
            "y": box.get("y"),
            "width": box.get("width"),
            "height": box.get("height"),
        })

text_df = pd.DataFrame(text_rows)

print(f"Text blocks: {len(text_df)}")
text_df.head()

## Make a table dataset

This creates one row for every table in NDV1. The table's original `content` is kept as text.

In [ ]:
table_rows = []

for page in ndv1["pages"]:
    for table in page.get("tables", []):
        table_rows.append({
            "page": page["number"],
            "table_id": table["id"],
            "content": table["content"],
        })

table_df = pd.DataFrame(table_rows)

print(f"Tables: {len(table_df)}")
table_df.head()

## Flatten NDV1 all the way to leaf values

This recursive transformation visits every nested dictionary and list. Each row contains the complete path to one leaf value, such as `pages[0].textBlocks[0].boundingBox.x`.

In [ ]:
def flatten_to_leaves(value, path=""):
    """Yield one record for every leaf value in a nested JSON object."""
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}" if path else str(key)
            yield from flatten_to_leaves(child, child_path)
    elif isinstance(value, list):
        for index, child in enumerate(value):
            yield from flatten_to_leaves(child, f"{path}[{index}]")
    else:
        yield {
            "path": path,
            "value": value,
            "value_type": type(value).__name__,
        }

leaf_df = pd.DataFrame(flatten_to_leaves(ndv1))

print(f"Leaf values: {len(leaf_df)}")
leaf_df.head(20)

## Create named columns for each NDV1 section

Because each page can contain many tables and text blocks, we create separate flat tables. This keeps the aliases readable and avoids hundreds of unrelated columns in one giant table.

In [ ]:
page_rows = []
table_alias_rows = []
text_block_alias_rows = []
visual_region_rows = []

for page_index, page in enumerate(ndv1.get("pages", [])):
    page_number = page["number"]

    page_rows.append({
        "artifact_id": ndv1["artifactId"],
        "page_index": page_index,
        "page_number": page_number,
        "page_width": page.get("width"),
        "page_height": page.get("height"),
    })

    for table_index, table in enumerate(page.get("tables", [])):
        box = table.get("boundingBox", {})
        table_alias_rows.append({
            "artifact_id": ndv1["artifactId"],
            "page_number": page_number,
            "table_index": table_index,
            "table_id": table["id"],
            "table_content": table.get("content"),
            "table_bounding_box_x": box.get("x"),
            "table_bounding_box_y": box.get("y"),
            "table_bounding_box_width": box.get("width"),
            "table_bounding_box_height": box.get("height"),
        })

    for block_index, block in enumerate(page.get("textBlocks", [])):
        box = block.get("boundingBox", {})
        text_block_alias_rows.append({
            "artifact_id": ndv1["artifactId"],
            "page_number": page_number,
            "text_block_index": block_index,
            "text_block_id": block["id"],
            "text_block_kind": block.get("kind"),
            "text_block_text": block.get("text"),
            "text_block_confidence": block.get("confidence"),
            "text_block_bounding_box_x": box.get("x"),
            "text_block_bounding_box_y": box.get("y"),
            "text_block_bounding_box_width": box.get("width"),
            "text_block_bounding_box_height": box.get("height"),
        })

    for region_index, region in enumerate(page.get("visualRegions", [])):
        box = region.get("boundingBox", {})
        visual_region_rows.append({
            "artifact_id": ndv1["artifactId"],
            "page_number": page_number,
            "visual_region_index": region_index,
            "visual_region_id": region["id"],
            "visual_region_label": region.get("label"),
            "visual_region_asset_ref": region.get("assetRef"),
            "visual_region_bounding_box_x": box.get("x"),
            "visual_region_bounding_box_y": box.get("y"),
            "visual_region_bounding_box_width": box.get("width"),
            "visual_region_bounding_box_height": box.get("height"),
        })

pages_alias_df = pd.DataFrame(page_rows)
tables_alias_df = pd.DataFrame(table_alias_rows)
text_blocks_alias_df = pd.DataFrame(text_block_alias_rows)
visual_regions_df = pd.DataFrame(visual_region_rows)

print("Pages columns:", list(pages_alias_df.columns))
print("Tables columns:", list(tables_alias_df.columns))
print("Text-block columns:", list(text_blocks_alias_df.columns))
print("Visual-region columns:", list(visual_regions_df.columns))

text_blocks_alias_df.head()

## Combine all aliased records into one CSV

The combined table uses one row per page, table, text block, or visual region. The `record_type` column tells us what each row represents. Fields that do not apply to a record type are left blank.

In [ ]:
combined_df = pd.concat([
    pages_alias_df.assign(record_type="page"),
    tables_alias_df.assign(record_type="table"),
    text_blocks_alias_df.assign(record_type="text_block"),
    visual_regions_df.assign(record_type="visual_region"),
], ignore_index=True, sort=False)

# Put the type column first so the row meaning is immediately visible.
record_type = combined_df.pop("record_type")
combined_df.insert(0, "record_type", record_type)

print(f"Combined rows: {len(combined_df)}")
combined_df.head()

## Optional: save the pandas tables

These CSV files are derived views. The original NDV1 JSON remains unchanged.

In [ ]:
output_dir = repository_root / "dataset" / "derived"
output_dir.mkdir(parents=True, exist_ok=True)

text_df.to_csv(output_dir / "text_blocks.csv", index=False)
table_df.to_csv(output_dir / "tables.csv", index=False)
leaf_df.to_csv(output_dir / "ndv1_leaf_values.csv", index=False)
pages_alias_df.to_csv(output_dir / "pages_aliases.csv", index=False)
tables_alias_df.to_csv(output_dir / "tables_aliases.csv", index=False)
text_blocks_alias_df.to_csv(output_dir / "text_blocks_aliases.csv", index=False)
visual_regions_df.to_csv(output_dir / "visual_regions.csv", index=False)
combined_df.to_csv(output_dir / "ndv1_combined.csv", index=False)

print(f"Saved CSV files to: {output_dir.resolve()}")